In [1]:
import torch
from transformers import BitsAndBytesConfig

config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
)

/home/yang/projects/nn-zero-to-hero/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from transformers import AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained("mistralai/Mistral-7B-v0.1", quantization_config=config)

Loading weights:   0%|          | 1/291 [00:01<07:19,  1.52s/it]/home/yang/projects/nn-zero-to-hero/.venv/lib/python3.11/site-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
Loading weights: 100%|██████████| 291/291 [00:06<00:00, 45.24it/s] 


In [3]:
from peft import prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

In [4]:
from peft import LoraConfig

config = LoraConfig(
    r=16,
    lora_alpha=8,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

In [5]:
from peft import get_peft_model

model = get_peft_model(model, config)

In [7]:
from datasets import load_dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          TrainingArguments, Trainer, DataCollatorWithPadding)
from peft import LoraConfig, get_peft_model, TaskType
import numpy as np, evaluate

ckpt = "distilbert-base-uncased"
ds = load_dataset("nyu-mll/glue", "sst2")
tok = AutoTokenizer.from_pretrained(ckpt)
ds_tok = ds.map(lambda b: tok(b["sentence"], truncation=True), batched=True)
model = AutoModelForSequenceClassification.from_pretrained(ckpt, num_labels=2)

# 关键：套上 LoRA
lora = LoraConfig(task_type=TaskType.SEQ_CLS, r=8, lora_alpha=16,
                  lora_dropout=0.1, target_modules=["q_lin", "v_lin"])
model = get_peft_model(model, lora)
model.print_trainable_parameters()   # ⭐ 看可训练参数占比（通常 <1%）

metric = evaluate.load("glue", "sst2")
def cm(p): return metric.compute(predictions=np.argmax(p.predictions,1), references=p.label_ids)
args = TrainingArguments("lora-out", eval_strategy="epoch", num_train_epochs=2,
        per_device_train_batch_size=32, learning_rate=2e-4,  # LoRA 用更大 lr
        report_to="wandb", run_name="distilbert-sst2-lora")
Trainer(model, args, train_dataset=ds_tok["train"], eval_dataset=ds_tok["validation"],
        data_collator=DataCollatorWithPadding(tok), processing_class=tok,
        compute_metrics=cm).train()

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 1729.37it/s]
[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


trainable params: 739,586 || all params: 67,694,596 || trainable%: 1.0925


wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /home/yang/.netrc.
wandb: Currently logged in as: ezrealzheng11 (ezrealzheng11-new-york-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


[transformers] `use_return_dict` is deprecated! Use `return_dict` instead!


Epoch,Training Loss,Validation Loss,Accuracy
1,0.258365,0.273013,0.884174
2,0.222464,0.280632,0.887615


TrainOutput(global_step=4210, training_loss=0.2573865740995792, metrics={'train_runtime': 77.5209, 'train_samples_per_second': 1737.57, 'train_steps_per_second': 54.308, 'total_flos': 1425856614633408.0, 'train_loss': 0.2573865740995792, 'epoch': 2.0})